# 21 — Per-Client Model Specialization

Fine-tunes a tenant-specific SBERT model from accumulated corrections.
Triggered when a tenant has 100+ corrections.
Registered in MLflow as `sbert-ifrs-matcher-{tenant_id}`.


In [ ]:
# Cell 1: Setup
import sys, os, json
sys.path.insert(0, "/content/drive/MyDrive/numera-ml")

from pathlib import Path
from collections import Counter
import httpx
import mlflow
from google.colab import drive
drive.mount("/content/drive")

import yaml
with open("/content/drive/MyDrive/numera-ml/configs/training_config.yaml") as f:
    config = yaml.safe_load(f)

mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])

MODELS_DIR = Path(config["drive"]["models_dir"])
EXPORTED_DIR = Path(config["drive"]["exported_dir"])

ML_SERVICE_URL = os.environ.get("ML_SERVICE_URL", "http://localhost:8002")

# Set the tenant ID to specialize
TENANT_ID = "tenant-123"  # Change this per run


In [ ]:
# Cell 2: Download tenant-specific feedback

resp = httpx.get(
    f"{ML_SERVICE_URL}/api/ml/feedback/export",
    params={"tenant_id": TENANT_ID},
    timeout=60,
)
resp.raise_for_status()
records = resp.json()["records"]

print(f"Tenant {TENANT_ID}: {len(records)} feedback records")

if len(records) < 100:
    print(f"Not enough corrections ({len(records)} < 100). Exiting.")


In [ ]:
# Cell 3: Load global Production model and fine-tune

from sentence_transformers import (
    SentenceTransformer, InputExample, losses,
    SentenceTransformerTrainer, SentenceTransformerTrainingArguments,
)

# Load global model
try:
    prod_path = mlflow.artifacts.download_artifacts(
        artifact_uri="models:/sbert-ifrs-matcher/Production",
        dst_path=str(MODELS_DIR / "sbert_prod_base"),
    )
    model = SentenceTransformer(prod_path)
except Exception:
    model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate training pairs from corrections
pairs = []
correction_counts = Counter(
    (r["source_text"], r.get("corrected_item_label", r["corrected_item_id"]))
    for r in records if r.get("correction_type") == "REMAPPED"
)

for (src, tgt), count in correction_counts.items():
    if count >= 2:  # Lower threshold for client models
        pairs.append(InputExample(texts=[src, tgt], label=1.0))

print(f"Training pairs: {len(pairs)}")

# Fine-tune
with mlflow.start_run(run_name=f"sbert-client-{TENANT_ID}"):
    mlflow.log_params({
        "tenant_id": TENANT_ID,
        "total_records": len(records),
        "training_pairs": len(pairs),
    })

    training_args = SentenceTransformerTrainingArguments(
        output_dir=str(MODELS_DIR / "checkpoints" / f"sbert_client_{TENANT_ID}"),
        num_train_epochs=3,
        per_device_train_batch_size=16,
        learning_rate=1e-5,
        fp16=True,
    )

    trainer = SentenceTransformerTrainer(
        model=model, args=training_args,
        train_dataset=pairs,
        loss=losses.CosineSimilarityLoss(model),
    )
    trainer.train()


In [ ]:
# Cell 4: Register as tenant-specific model

export_path = EXPORTED_DIR / f"sbert_client_{TENANT_ID}"
model.save(str(export_path))

model_name = f"sbert-ifrs-matcher-{TENANT_ID}"

mlflow.log_artifact(str(export_path), "model")
result = mlflow.register_model(
    f"runs:/{mlflow.active_run().info.run_id}/model",
    model_name,
)

from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(
    name=model_name,
    version=result.version,
    stage="Production",
)

print(f"Client model registered: {model_name} v{result.version}")
print(f"ml-service will auto-load this model for tenant {TENANT_ID}")
